In [1]:
import random
import numpy as np
from tqdm import tqdm

In [23]:
class Maze:
    def __init__(self):
        self.maze = [
            "OOOOOOOCG",
            "OOCOOOOCO",
            "SOCOOOOCO",
            "OOCOOOOOO",
            "OOOOOCOOO",
            "OOOOOOOOO",
        ]
        self.starting_state = (2,0)
        self.goal_state = (0, 8)

        self.number_of_steps = 0
        self.current_state = self.starting_state

        self.terminated = 0
        self.truncated = 0

        self.terminal_states = [(1,2), (2, 2), (3, 2), (0, 7), (1, 7), (2, 7), (4, 5), (0,8)]

    def get_action_space(self):
        return ["up", "down", "left", "right"]

    def print_maze(self):
        print(self.maze)

    def change_state(self, action):

        new_state = (0, 0)

        if action == "up":
            if self.current_state[0] == 0:
                return
            new_state = (self.current_state[0] - 1, self.current_state[1])
        elif action == "down":
            if self.current_state[0] == 5:
                return
            new_state = (self.current_state[0] + 1, self.current_state[1])
        elif action == "left":
            if self.current_state[1] == 0:
                return
            new_state = (self.current_state[0], self.current_state[1] - 1)
        elif action == "right":
            if self.current_state[1] == 8:
                return

            new_state = (self.current_state[0], self.current_state[1] + 1)

        if (
            self.maze[new_state[0]][new_state[1]] == "O"
            or self.maze[new_state[0]][new_state[1]] == "S"
        ):
            self.current_state = new_state
        elif self.maze[new_state[0]][new_state[1]] == "G":
            self.current_state = new_state

    def step(self, action):

        if self.terminated or self.truncated:
            observation = self.current_state
            reward = 0
            return observation, reward, self.terminated, self.truncated

        self.number_of_steps += 1

        self.change_state(action)

        observation = self.current_state

        reward = 0
        if self.maze[self.current_state[0]][self.current_state[1]] == "G":
            reward = 1

            self.terminated  = 1
        else:
            reward = 0

        if self.number_of_steps == 100000:
            self.truncated = 1

        return observation, reward, self.terminated, self.truncated

    def reset(self):
        self.number_of_steps = 0
        self.current_state = self.starting_state

        self.terminated = 0
        self.truncated = 0

In [25]:
class ChangingMaze(Maze):
    def __init__(self):
        super().__init__()
        self.maze = [
            "OOOOOOOOG",
            "OOOOOOOOO",
            "OOOOOOOOO",
            "CCCCCCCCO",
            "OOOOOOOOO",
            "OOOSOOOOO",
        ]

        self.starting_state = (5,3)
        self.goal_state = (0, 8)

        self.number_of_steps = 0
        self.current_state = self.starting_state

        self.terminated = 0
        self.truncated = 0

        self.terminal_states = [
            (3, 0),
            (3, 1),
            (3, 2),
            (3, 3),
            (3, 4),
            (3, 5),
            (3, 6),
            (3, 7),
        ]

        self.total_number_of_steps = 0
    def reset(self):
        self.total_number_of_steps += self.number_of_steps

        if self.total_number_of_steps > 1000:
            self.maze = [
                "OOOOOOOOG",
                "OOOOOOOOO",
                "OOOOOOOOO",
                "OCCCCCCCC",
                "OOOOOOOOO",
                "OOOSOOOOO",
            ]

        self.number_of_steps = 0
        self.current_state = self.starting_state

        self.terminated = 0
        self.truncated = 0

In [26]:
maze = ChangingMaze()

In [27]:
class Dyna_Q:
    def __init__(self, env, n, number_of_episodes = 100):
        self.states_and_actions_taken = {}
        self.env = env
        self.n = n

        self.Q = {}
        self.Model = {}

        self.initialize_Q_and_Model()

        self.number_of_episodes = number_of_episodes

        self.trajectories = []

    def initialize_Q_and_Model(self):

        for row in range(6):
            for col in range(9):
                self.Q[(row, col)] = {"up": 0, "down": 0, "left": 0, "right": 0}
                self.Model[(row, col)] = {"up": (0,0), "down": (0,0), "left": (0,0), "right": (0,0)}

    def get_argmax(self, S):
        state_dict = self.Q[S]

        max_value = max(state_dict.values())
        return [action for action, value in state_dict.items() if value == max_value]

    def get_max(self, S):
        state_dict = self.Q[S]

        action = max(state_dict, key=state_dict.get)
        return self.Q[S][action]

    def get_action(self, epsilon, S):
        action = random.choice(self.env.get_action_space())
        if random.random() > epsilon:
            possible_actions = self.get_argmax(S)
            action = random.choice(possible_actions)

        return action

    def update_state_action_collection(self, S, A):
        if S not in self.states_and_actions_taken:
            self.states_and_actions_taken[S] = []

        if A not in self.states_and_actions_taken[S]:
            self.states_and_actions_taken[S].append(A)

    def get_policy(self):
        policy = {}
        terminal_states = self.env.terminal_states
        for S in self.Q:

            if S in terminal_states:
                policy[S] = "Terminal"
                continue

            A = self.get_argmax(S)[0]

            policy[S] = A
        return policy

    def do_planning_updates(self):
        for _ in range(self.n):

            S = random.choice(list(self.states_and_actions_taken.keys()))
            A = random.choice(self.states_and_actions_taken[S])

            R, S_prime = self.Model[S][A]


            self.Q[S][A] = self.Q[S][A] + 0.1 * (
                R + 0.9999999999999999 * self.get_max(S_prime) - self.Q[S][A]
            )

    def do_one_dyna_Q_iteration(self):
        self.env.reset()

        S = self.env.current_state
        R = 0
        S_prime = self.env.current_state
        terminated = 0
        truncated = 0

        trajectory = []

        while not terminated and not truncated:
            S = S_prime
            trajectory.append(S)
            A = self.get_action(0.2, S)

            self.update_state_action_collection(S, A)

            S_prime, R, terminated, truncated  = self.env.step(A)

            self.Q[S][A] = self.Q[S][A] + 0.1 * (
                R + 0.9999999999999999 * self.get_max(S_prime) - self.Q[S][A]
            )

            self.Model[S][A] = (R, S_prime)

            self.do_planning_updates()

        self.trajectories.append(trajectory)

    def do_dyna_Q(self):
        for i in tqdm(range(self.number_of_episodes)):
            self.do_one_dyna_Q_iteration()

In [28]:
dyna = Dyna_Q(maze, n=50, number_of_episodes=1000)

In [29]:
dyna.do_dyna_Q()

  4%|▍         | 40/1000 [01:52<45:10,  2.82s/it]  


KeyboardInterrupt: 

In [21]:
policy = dyna.get_policy()

In [22]:
policy

{(0, 0): 'right',
 (0, 1): 'right',
 (0, 2): 'right',
 (0, 3): 'right',
 (0, 4): 'right',
 (0, 5): 'right',
 (0, 6): 'right',
 (0, 7): 'right',
 (0, 8): 'up',
 (1, 0): 'up',
 (1, 1): 'up',
 (1, 2): 'up',
 (1, 3): 'up',
 (1, 4): 'up',
 (1, 5): 'up',
 (1, 6): 'up',
 (1, 7): 'up',
 (1, 8): 'up',
 (2, 0): 'up',
 (2, 1): 'up',
 (2, 2): 'up',
 (2, 3): 'up',
 (2, 4): 'up',
 (2, 5): 'up',
 (2, 6): 'up',
 (2, 7): 'up',
 (2, 8): 'up',
 (3, 0): 'Terminal',
 (3, 1): 'Terminal',
 (3, 2): 'Terminal',
 (3, 3): 'Terminal',
 (3, 4): 'Terminal',
 (3, 5): 'Terminal',
 (3, 6): 'Terminal',
 (3, 7): 'Terminal',
 (3, 8): 'up',
 (4, 0): 'right',
 (4, 1): 'right',
 (4, 2): 'right',
 (4, 3): 'right',
 (4, 4): 'right',
 (4, 5): 'right',
 (4, 6): 'right',
 (4, 7): 'right',
 (4, 8): 'up',
 (5, 0): 'up',
 (5, 1): 'up',
 (5, 2): 'up',
 (5, 3): 'up',
 (5, 4): 'up',
 (5, 5): 'up',
 (5, 6): 'up',
 (5, 7): 'up',
 (5, 8): 'up'}

In [370]:
class PlotMaze():
    def __init__(self, maze, policy):
        self.maze = maze
        self.policy = policy